# 02 — MDM2 pIC50 PaDEL 1D/2D and Morgan Feature Preparation

## Overview

This notebook prepares the molecular feature representation used for downstream MDM2 pIC50 QSAR modeling.

PaDEL 1D/2D descriptors were generated previously using a separate batch-processing workflow and are therefore loaded from the validated descriptor output rather than recalculated.

Morgan circular fingerprints are generated from the standardized molecular structures using RDKit.

The final feature representation combines:

1. **PaDEL 1D/2D molecular descriptors**
2. **Morgan circular fingerprints**

Only molecules with valid PaDEL descriptors and corresponding experimental pIC50 values are retained.

No scaffold splitting, imputation, variance filtering, feature selection, or model fitting is performed in this notebook. These operations are performed in Notebook 03 using training-set-derived transformations only.

## Step 1 — Import Libraries and Define Paths

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator


# ============================================================
# Project configuration
# ============================================================

PROJECT_ROOT = Path(
    "/Users/maryta/Downloads/mdm2_Reinvent_4_final"
)


MAPPING_FILE = (
    PROJECT_ROOT
    / "data"
    / "padel"
    / "mdm2_pic50_Pubchem"
    / "input"
    / "mdm2_padel_row_mapping.csv"
)


DESCRIPTOR_FILE = (
    PROJECT_ROOT
    / "data"
    / "padel"
    / "mdm2_pic50_1D_2D"
    / "output"
    / "mdm2_1d2d_descriptors_padel_eligible.csv"
)


EXCLUDED_FILE = (
    PROJECT_ROOT
    / "data"
    / "padel"
    / "mdm2_pic50_1D_2D"
    / "input"
    / "mdm2_padel_excluded_large_or_invalid.csv"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "qsar_features"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


OUTPUT_FILE = (
    OUTPUT_DIR
    / "mdm2_padel_1d2d_morgan_features.csv"
)


print("Mapping file:")
print(MAPPING_FILE)

print("\nPaDEL descriptor file:")
print(DESCRIPTOR_FILE)

print("\nExcluded molecules file:")
print(EXCLUDED_FILE)

print("\nFinal output:")
print(OUTPUT_FILE)

Mapping file:
/Users/maryta/Downloads/mdm2_Reinvent_4_final/data/padel/mdm2_pic50_Pubchem/input/mdm2_padel_row_mapping.csv

PaDEL descriptor file:
/Users/maryta/Downloads/mdm2_Reinvent_4_final/data/padel/mdm2_pic50_1D_2D/output/mdm2_1d2d_descriptors_padel_eligible.csv

Excluded molecules file:
/Users/maryta/Downloads/mdm2_Reinvent_4_final/data/padel/mdm2_pic50_1D_2D/input/mdm2_padel_excluded_large_or_invalid.csv

Final output:
/Users/maryta/Downloads/mdm2_Reinvent_4_final/data/processed/qsar_features/mdm2_padel_1d2d_morgan_features.csv


## Step 2 — Load and Validate Precomputed PaDEL 1D/2D Descriptors

PaDEL 1D/2D molecular descriptors were generated previously in a separate descriptor-generation workflow and are not recalculated in this notebook.

The earlier workflow processed the PaDEL-eligible MDM2 molecules in small resumable batches because descriptor generation was computationally expensive and time-consuming. The completed batch outputs were combined into a final descriptor table and validated for:

- expected molecule counts,
- unique molecule identifiers,
- missing molecule IDs,
- unexpected molecule IDs,
- duplicate identifiers,
- preservation of molecule order.

The validated PaDEL descriptor table was saved as:

`mdm2_1d2d_descriptors_padel_eligible.csv`

Notebook 02 therefore reuses this precomputed descriptor table rather than repeating the full PaDEL calculation.

The saved descriptors are loaded and aligned with the corresponding standardized SMILES and experimental pIC50 values using the PaDEL molecule identifier.

Morgan fingerprints are generated separately in this notebook using RDKit because their calculation is comparatively fast.

This step therefore focuses on validating the existing PaDEL output and preparing it for integration with the newly generated Morgan fingerprint features.

In [8]:
# ============================================================
# Step 2 — Load and Validate PaDEL Inputs
# ============================================================

for file_path in [
    MAPPING_FILE,
    DESCRIPTOR_FILE,
    EXCLUDED_FILE,
]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )


mapping = pd.read_csv(
    MAPPING_FILE,
    low_memory=False,
)

descriptors = pd.read_csv(
    DESCRIPTOR_FILE,
    low_memory=False,
)

excluded = pd.read_csv(
    EXCLUDED_FILE,
    low_memory=False,
)


print("Mapping shape:", mapping.shape)
print("Descriptor shape:", descriptors.shape)
print("Excluded shape:", excluded.shape)

Mapping shape: (4146, 4)
Descriptor shape: (4048, 1445)
Excluded shape: (98, 6)


## Step 3 — Validate Molecule Identifiers and Descriptor Coverage

In [10]:
# ============================================================
# Step 3 — Validate IDs and Descriptor Coverage
# ============================================================

required_mapping_columns = {
    "padel_molecule_id",
    "standardized_smiles",
    "pIC50",
}

required_descriptor_columns = {
    "Name"
}


missing_mapping_columns = (
    required_mapping_columns
    - set(mapping.columns)
)

missing_descriptor_columns = (
    required_descriptor_columns
    - set(descriptors.columns)
)


if missing_mapping_columns:
    raise KeyError(
        "Missing mapping columns: "
        f"{sorted(missing_mapping_columns)}"
    )


if missing_descriptor_columns:
    raise KeyError(
        "Missing descriptor columns: "
        f"{sorted(missing_descriptor_columns)}"
    )


mapping["padel_molecule_id"] = (
    mapping["padel_molecule_id"]
    .astype(str)
    .str.strip()
)

descriptors["Name"] = (
    descriptors["Name"]
    .astype(str)
    .str.strip()
)


mapping_ids = set(
    mapping["padel_molecule_id"]
)

descriptor_ids = set(
    descriptors["Name"]
)


missing_ids = (
    mapping_ids
    - descriptor_ids
)

unexpected_ids = (
    descriptor_ids
    - mapping_ids
)


mapping_duplicates = (
    mapping["padel_molecule_id"]
    .duplicated()
    .sum()
)

descriptor_duplicates = (
    descriptors["Name"]
    .duplicated()
    .sum()
)


descriptor_rate = (
    100
    * len(descriptor_ids)
    / len(mapping_ids)
)


print("=" * 60)
print("PaDEL DATASET SUMMARY")
print("=" * 60)

print(
    "Mapped molecules:",
    len(mapping)
)

print(
    "Unique mapped IDs:",
    len(mapping_ids)
)

print(
    "Descriptor rows:",
    len(descriptors)
)

print(
    "Unique descriptor IDs:",
    len(descriptor_ids)
)

print(
    "Excluded molecules:",
    len(excluded)
)

print(
    "Mapped IDs without descriptors:",
    len(missing_ids)
)

print(
    "Unexpected descriptor IDs:",
    len(unexpected_ids)
)

print(
    "Duplicate mapping IDs:",
    mapping_duplicates
)

print(
    "Duplicate descriptor IDs:",
    descriptor_duplicates
)

print(
    f"Descriptor coverage: {descriptor_rate:.2f}%"
)

PaDEL DATASET SUMMARY
Mapped molecules: 4146
Unique mapped IDs: 4146
Descriptor rows: 4048
Unique descriptor IDs: 4048
Excluded molecules: 98
Mapped IDs without descriptors: 98
Unexpected descriptor IDs: 0
Duplicate mapping IDs: 0
Duplicate descriptor IDs: 0
Descriptor coverage: 97.64%


## Step 4 — Align PaDEL Descriptors with Molecular and Activity Data

The PaDEL descriptor table is linked to the mapping table using the PaDEL molecule identifier.

An inner one-to-one merge is used so that only molecules with valid PaDEL descriptors, standardized SMILES, and experimental pIC50 values are retained.

In [11]:
# ============================================================
# Step 4 — Align PaDEL Descriptors with Molecular Data
# ============================================================

df_padel_matched = descriptors.merge(
    mapping[
        [
            "padel_molecule_id",
            "standardized_smiles",
            "pIC50",
        ]
    ],
    left_on="Name",
    right_on="padel_molecule_id",
    how="inner",
    validate="one_to_one",
)


print(
    "Matched dataset shape:",
    df_padel_matched.shape
)

print(
    "Descriptor rows not matched:",
    len(descriptors)
    - len(df_padel_matched)
)

print(
    "Mapping rows without descriptors:",
    len(mapping)
    - len(df_padel_matched)
)

print(
    "Missing pIC50:",
    df_padel_matched["pIC50"]
    .isna()
    .sum()
)

print(
    "Missing standardized SMILES:",
    df_padel_matched["standardized_smiles"]
    .isna()
    .sum()
)


display(
    df_padel_matched[
        [
            "Name",
            "padel_molecule_id",
            "standardized_smiles",
            "pIC50",
        ]
    ].head()
)

Matched dataset shape: (4048, 1448)
Descriptor rows not matched: 0
Mapping rows without descriptors: 98
Missing pIC50: 0
Missing standardized SMILES: 0


,Name,padel_molecule_id,standardized_smiles,pIC50
0,MDM2_000000,MDM2_000000,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,7.465466
1,MDM2_000001,MDM2_000001,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,8.116422
2,MDM2_000002,MDM2_000002,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,7.958607
3,MDM2_000003,MDM2_000003,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,8.283997
4,MDM2_000004,MDM2_000004,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,8.283997


## Step 5 — Generate Morgan Fingerprints

Morgan circular fingerprints are generated for the same molecules retained in the PaDEL-matched dataset.

The fingerprints use:

- radius = 2
- fingerprint size = 2,048 bits

Generating Morgan fingerprints after PaDEL alignment ensures that both feature representations refer to exactly the same molecules and row order.

In [12]:
# ============================================================
# Step 5 — Generate Morgan Fingerprints
# ============================================================

MORGAN_RADIUS = 2
MORGAN_N_BITS = 2048


morgan_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=MORGAN_RADIUS,
        fpSize=MORGAN_N_BITS,
    )
)


def smiles_to_morgan(smiles):
    """
    Convert a standardized SMILES string into a
    2048-bit Morgan fingerprint.
    """

    if pd.isna(smiles):
        return None

    smiles = str(smiles).strip()

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:
        return None

    fingerprint = (
        morgan_generator.GetFingerprint(
            mol
        )
    )

    fingerprint_array = np.zeros(
        MORGAN_N_BITS,
        dtype=np.uint8,
    )

    DataStructs.ConvertToNumpyArray(
        fingerprint,
        fingerprint_array,
    )

    return fingerprint_array


fingerprint_list = (
    df_padel_matched[
        "standardized_smiles"
    ]
    .apply(smiles_to_morgan)
    .tolist()
)


failed_fingerprints = sum(
    fingerprint is None
    for fingerprint in fingerprint_list
)


print(
    "Failed fingerprint calculations:",
    failed_fingerprints
)


if failed_fingerprints > 0:
    raise ValueError(
        "Some Morgan fingerprints could not be generated."
    )


X_morgan = np.vstack(
    fingerprint_list
)


print(
    "Morgan matrix shape:",
    X_morgan.shape
)

print(
    "Morgan radius:",
    MORGAN_RADIUS
)

print(
    "Morgan fingerprint size:",
    MORGAN_N_BITS
)

Failed fingerprint calculations: 0
Morgan matrix shape: (4048, 2048)
Morgan radius: 2
Morgan fingerprint size: 2048


## Step 6 — Create the Morgan Feature Table

In [13]:
# ============================================================
# Step 6 — Create Morgan Feature Table
# ============================================================

morgan_feature_names = [
    f"Morgan_r{MORGAN_RADIUS}_bit_{bit_number:04d}"
    for bit_number in range(
        MORGAN_N_BITS
    )
]


df_morgan = pd.DataFrame(
    X_morgan,
    columns=morgan_feature_names,
    index=df_padel_matched.index,
)


print(
    "Morgan feature table shape:",
    df_morgan.shape
)


display(
    df_morgan.iloc[
        :5,
        :10
    ]
)

Morgan feature table shape: (4048, 2048)


,Morgan_r2_bit_0000,Morgan_r2_bit_0001,Morgan_r2_bit_0002,Morgan_r2_bit_0003,Morgan_r2_bit_0004,Morgan_r2_bit_0005,Morgan_r2_bit_0006,Morgan_r2_bit_0007,Morgan_r2_bit_0008,Morgan_r2_bit_0009
0,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


## Step 7 — Combine PaDEL and Morgan Features

The complete PaDEL descriptor table and Morgan fingerprint matrix are combined while preserving molecule identifiers, standardized SMILES, and experimental pIC50 values.

No feature filtering is performed at this stage. The complete feature matrix is retained so that training-dependent preprocessing can be performed after scaffold splitting in Notebook 03.

In [14]:
# ============================================================
# Step 7 — Combine PaDEL and Morgan Features
# ============================================================

metadata_columns = [
    "Name",
    "padel_molecule_id",
    "standardized_smiles",
    "pIC50",
]


padel_descriptor_columns = [
    column
    for column in descriptors.columns
    if column != "Name"
]


df_features = pd.concat(
    [
        df_padel_matched[
            metadata_columns
        ].reset_index(
            drop=True
        ),

        df_padel_matched[
            padel_descriptor_columns
        ].reset_index(
            drop=True
        ),

        df_morgan.reset_index(
            drop=True
        ),
    ],
    axis=1,
)


print(
    "Final combined feature table shape:",
    df_features.shape
)

print(
    "PaDEL descriptor count:",
    len(padel_descriptor_columns)
)

print(
    "Morgan feature count:",
    len(morgan_feature_names)
)

print(
    "Total molecular feature count:",
    len(padel_descriptor_columns)
    + len(morgan_feature_names)
)


display(
    df_features.iloc[
        :5,
        :12
    ]
)

Final combined feature table shape: (4048, 3496)
PaDEL descriptor count: 1444
Morgan feature count: 2048
Total molecular feature count: 3492


,Name,padel_molecule_id,standardized_smiles,pIC50,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom
0,MDM2_000000,MDM2_000000,C#CCC1CN(c2nc3nc(C(=O)O)nc(N[C@H](C)C4CCC4)c3n...,7.465466,1,1.0515,1.105652,83.1599,77.533997,15,16,68
1,MDM2_000001,MDM2_000001,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)cc(F)c3[C@]2(OC...,8.116422,0,0.6833,0.466899,70.9986,80.941997,18,18,67
2,MDM2_000002,MDM2_000002,C#Cc1ccc(CN2C(=O)c3cc(C(C)(C)O)ccc3[C@]2(OCC2(...,7.958607,0,0.2036,0.041453,70.1261,81.051790,18,18,67
3,MDM2_000003,MDM2_000003,C#Cc1ccc(CN2C(=O)c3cc(C(C)(O)CO)cc(F)c3[C@]2(O...,8.283997,0,-0.2049,0.041984,72.5429,81.743997,18,18,68
4,MDM2_000004,MDM2_000004,C#Cc1ccc(CN2C(=O)c3cc([C@@](C)(O)CO)cc(F)c3[C@...,8.283997,0,-0.2049,0.041984,72.5429,81.743997,18,18,68


## Step 8 — Validate the Combined Feature Table

In [15]:
# ============================================================
# Step 8 — Validate Combined Feature Table
# ============================================================

if len(df_features) != len(
    df_padel_matched
):
    raise ValueError(
        "Row count changed during feature combination."
    )


if not (
    df_features["Name"]
    .astype(str)
    .tolist()
    ==
    df_padel_matched["Name"]
    .astype(str)
    .tolist()
):
    raise ValueError(
        "Molecule ordering changed during feature combination."
    )


print(
    "Rows:",
    len(df_features)
)

print(
    "Unique molecule IDs:",
    df_features["Name"].nunique()
)

print(
    "Duplicate molecule IDs:",
    df_features["Name"]
    .duplicated()
    .sum()
)

print(
    "Missing pIC50:",
    df_features["pIC50"]
    .isna()
    .sum()
)

print(
    "Missing SMILES:",
    df_features[
        "standardized_smiles"
    ]
    .isna()
    .sum()
)


print(
    "\nCombined feature table validation passed."
)

Rows: 4048
Unique molecule IDs: 4048
Duplicate molecule IDs: 0
Missing pIC50: 0
Missing SMILES: 0

Combined feature table validation passed.


## Step 9 — Save the Complete PaDEL + Morgan Feature Dataset

The complete unfiltered molecular feature table is saved for downstream scaffold splitting and ExtraTrees modeling in Notebook 03.

In [16]:
# ============================================================
# Step 9 — Save Feature Dataset
# ============================================================

df_features.to_csv(
    OUTPUT_FILE,
    index=False,
)


print(
    "Saved combined feature dataset:"
)

print(
    OUTPUT_FILE
)

print(
    "\nSaved shape:",
    df_features.shape
)

print(
    "\nNotebook 02 completed successfully."
)

Saved combined feature dataset:
/Users/maryta/Downloads/mdm2_Reinvent_4_final/data/processed/qsar_features/mdm2_padel_1d2d_morgan_features.csv

Saved shape: (4048, 3496)

Notebook 02 completed successfully.


## Conclusion

The molecular feature dataset was successfully prepared for downstream MDM2 pIC50 QSAR modeling.

Precomputed PaDEL 1D/2D descriptors were aligned with the corresponding standardized molecular structures and experimental pIC50 values. Morgan fingerprints were generated using RDKit with a radius of 2 and a fingerprint size of 2,048 bits.

The resulting dataset contained 4,048 molecules and provided the combined molecular representation used for subsequent QSAR modeling.

No scaffold splitting, imputation, variance filtering, feature selection, or model fitting was performed at this stage. These data-dependent preprocessing and modeling steps were performed subsequently using the scaffold-separated training, validation, and test sets.